# 05.2 Floats, IEEE 754, and Precision

```python
>>> 0.1 + 0.2
0.30000000000000004
```

This is not a Python bug. It is not a rounding error in the usual sense. It is
the correct, inevitable result of storing decimal fractions in binary — and every
language using IEEE 754 gives the same answer.

Understanding **why** takes ten minutes and saves you from a career of confusing
financial bugs.

## Theory

### Floats are binary fractions

An integer stores exact digits. A float stores a number in **binary scientific
notation**, following the IEEE 754 double-precision standard:

```
value = sign × mantissa × 2^exponent
```

A 64-bit double splits those 64 bits as:

| Part | Bits | Holds |
|---|---|---|
| Sign | 1 | positive or negative |
| Exponent | 11 | the power of two |
| Mantissa | 52 | the significant digits |

That gives roughly **15–17 significant decimal digits**.

### Why 0.1 cannot be represented

In base 10, one third is `0.333...` — it never terminates. You cannot write it
exactly with finitely many decimal digits.

In base 2, **one tenth** has the same problem:

```
0.1 decimal = 0.0001100110011001100... binary   (repeating forever)
```

A float has 52 bits of mantissa, so the sequence is cut off. What gets stored is
the closest representable value — very slightly more than 0.1.

Add two such approximations and the tiny errors accumulate into something
visible.

### The rule that follows

> **Never compare floats with `==`.**

Use `math.isclose()`, or switch to `Decimal` when you need exact decimal
arithmetic (05.4).

### Which decimals *are* exact

A fraction is exactly representable in binary only when its denominator is a
**power of two**. So `0.5`, `0.25`, `0.125` and `0.75` are exact. `0.1`, `0.2`
and `0.3` are not.

In [ ]:
# The famous example.
result = 0.1 + 0.2

print("0.1 + 0.2 =", result)
print("Is it equal to 0.3?", result == 0.3)
print("The difference:", result - 0.3)

# Python prints the SHORTEST string that round-trips back to the same float,
# which is why 0.1 looks clean on its own.
print("")
print("Python's display of 0.1:", 0.1)
print("The actual stored value to 20 places:")
print("   ", format(0.1, ".20f"))

print("")
print("So 0.1 was never 0.1. It is the nearest float to it.")

In [ ]:
# See exactly what each float actually stores, as an exact fraction.
for value in [0.5, 0.25, 0.1, 0.2, 0.3]:
    numerator, denominator = value.as_integer_ratio()

    # Every float's denominator is a power of two. What distinguishes an
    # exact value is that the fraction stays SMALL - 1/2, 1/4 and so on.
    verdict = "EXACT" if len(str(numerator)) <= 3 else "approximated"

    print(f"{value}  ->  {verdict}")
    print(f"   stored as {numerator} / {denominator}")
    print(f"   to 20 places: {format(value, '.20f')}")
    print("")

print("0.5 and 0.25 have small power-of-two denominators, so they are exact.")
print("0.1, 0.2 and 0.3 need enormous fractions - each is only the")
print("nearest float to the decimal you wrote.")

## Seeing the bits

`struct` lets us look at the raw 64 bits, and split them into sign, exponent and
mantissa.

In [ ]:
import struct

def show_bits(value):
    """Print the IEEE 754 bit layout of a float."""
    # Pack as a big-endian double, then read the bits.
    packed = struct.pack(">d", value)
    bits = "".join(format(byte, "08b") for byte in packed)

    sign_bit = bits[0]
    exponent_bits = bits[1:12]
    mantissa_bits = bits[12:]

    print(f"{value}")
    print(f"   sign:     {sign_bit}         ({'negative' if sign_bit == '1' else 'positive'})")
    print(f"   exponent: {exponent_bits}   ({int(exponent_bits, 2)} raw, "
          f"{int(exponent_bits, 2) - 1023} actual)")
    print(f"   mantissa: {mantissa_bits[:26]}...")
    print("")


for value in [1.0, 0.5, 0.1, 2.0]:
    show_bits(value)

print("Notice 0.5 and 1.0 have mostly-zero mantissas - they are exact.")
print("0.1 has a repeating pattern that had to be cut off at 52 bits.")

## The limits of a double

In [ ]:
import sys

info = sys.float_info

print("What a Python float can do:")
print("   largest value:      ", info.max)
print("   smallest positive:  ", info.min)
print("   decimal digits:     ", info.dig, "reliable significant digits")
print("   mantissa bits:      ", info.mant_dig)
print("   epsilon:            ", info.epsilon)

print("")
print("Epsilon is the smallest gap from 1.0 to the next float:")
print("   1.0 + epsilon      =", 1.0 + info.epsilon)
print("   1.0 + epsilon/2    =", 1.0 + info.epsilon / 2, "<- too small to register")

# Beyond the maximum, floats become infinity rather than raising.
print("")
print("Overflow behaviour:")
print("   1e308 * 10  ->", 1e308 * 10)
print("   type:", type(1e308 * 10).__name__)

# Integers do not have this problem.
print("")
print("Compare with an int, which has no maximum:")
print("   10 ** 400 works fine and has", len(str(10 ** 400)), "digits")

## Precision loss in practice

Errors accumulate. Adding 0.1 ten times does not give 1.0.

In [ ]:
# Accumulating a small error.
running_total = 0.0
for _ in range(10):
    running_total += 0.1

print("Adding 0.1 ten times:")
print("   result:", running_total)
print("   equals 1.0?", running_total == 1.0)
print("   error:", running_total - 1.0)

# The error grows with more additions.
print("")
print("How the error grows:")
for count in [10, 100, 1000, 10000]:
    total = 0.0
    for _ in range(count):
        total += 0.1
    expected = count * 0.1
    print(f"   {count:>5} additions: error = {total - expected:.2e}")

# sum() with floats has the same issue, but math.fsum does not.
import math

values = [0.1] * 10
print("")
print("sum([0.1] * 10)      =", sum(values))
print("math.fsum([0.1] * 10)=", math.fsum(values), "<- exact")
print("")
print("math.fsum tracks intermediate precision and is worth knowing.")

In [ ]:
# Catastrophic cancellation: subtracting nearly-equal numbers
# destroys the significant digits.

large_a = 1e16
large_b = 1e16 + 1

print("Subtracting nearly equal large numbers:")
print("   1e16 + 1 - 1e16 =", large_b - large_a, "<- should be 1")

# The problem is that 1e16 + 1 cannot even be represented.
print("   is 1e16 + 1 == 1e16?", (1e16 + 1) == 1e16)

print("")
print("At this magnitude, consecutive floats are more than 1 apart:")
print("   1e16          ->", format(1e16, ".1f"))
print("   next float up ->", format(math.nextafter(1e16, math.inf), ".1f"))

print("")
print("This is why order matters when summing mixed magnitudes:")
small_first = 1e-10 + 1e-10 + 1e10
large_first = 1e10 + 1e-10 + 1e-10
print("   small values first:", small_first)
print("   large value first: ", large_first)
print("   same?", small_first == large_first)

## Comparing floats correctly

Never use `==`. Use `math.isclose()`, which handles both relative and absolute
tolerance.

In [ ]:
import math

value = 0.1 + 0.2

print("The wrong way:")
print("   0.1 + 0.2 == 0.3 ->", value == 0.3)

print("")
print("The right way:")
print("   math.isclose(0.1 + 0.2, 0.3) ->", math.isclose(value, 0.3))

# isclose takes two tolerances.
print("")
print("isclose(a, b, rel_tol=1e-09, abs_tol=0.0)")
print("   rel_tol: relative difference - scales with magnitude")
print("   abs_tol: absolute difference - needed when comparing to zero")

# Relative tolerance alone fails near zero.
tiny_a = 1e-18
tiny_b = 2e-18

print("")
print("Comparing tiny values near zero:")
print("   isclose(1e-18, 2e-18)                  ->", math.isclose(tiny_a, tiny_b))
print("   isclose(1e-18, 2e-18, abs_tol=1e-15)   ->",
      math.isclose(tiny_a, tiny_b, abs_tol=1e-15))

print("")
print("Rule: always pass abs_tol when either value might be zero.")

# A helper you can reuse.
def nearly_equal(first, second, tolerance=1e-9):
    """Compare two floats with both relative and absolute tolerance."""
    return math.isclose(first, second, rel_tol=tolerance, abs_tol=tolerance)


print("")
print("nearly_equal(0.1 + 0.2, 0.3) ->", nearly_equal(0.1 + 0.2, 0.3))
print("nearly_equal(0.0, 1e-15)     ->", nearly_equal(0.0, 1e-15))

## Special values: infinity and NaN

IEEE 754 defines three values that are not ordinary numbers.

In [ ]:
import math

positive_infinity = float("inf")
negative_infinity = float("-inf")
not_a_number = float("nan")

print("Special float values:")
print("   float('inf')  ->", positive_infinity)
print("   float('-inf') ->", negative_infinity)
print("   float('nan')  ->", not_a_number)

print("")
print("Infinity behaves sensibly in arithmetic:")
print("   inf + 1      ->", positive_infinity + 1)
print("   inf * 2      ->", positive_infinity * 2)
print("   1 / inf      ->", 1 / positive_infinity)
print("   inf > 10**100 ->", positive_infinity > 10 ** 100)

print("")
print("But some operations are undefined, producing NaN:")
print("   inf - inf    ->", positive_infinity - positive_infinity)
print("   inf / inf    ->", positive_infinity / positive_infinity)
print("   0 * inf      ->", 0 * positive_infinity)

In [ ]:
import math

not_a_number = float("nan")

# NaN is the only value that is not equal to itself.
print("NaN breaks the usual rules of equality:")
print("   nan == nan ->", not_a_number == not_a_number, "<- False!")
print("   nan != nan ->", not_a_number != not_a_number)
print("   nan < 1    ->", not_a_number < 1)
print("   nan > 1    ->", not_a_number > 1)

print("")
print("So never test for NaN with ==. Use math.isnan:")
print("   math.isnan(nan) ->", math.isnan(not_a_number))

# But `is` can still be True, because it compares identity.
same_nan = not_a_number
print("")
print("Identity still works, which causes surprises in containers:")
print("   nan is nan          ->", same_nan is not_a_number)
print("   nan in [nan]        ->", not_a_number in [not_a_number])
print("   (the `in` check uses `is` first, then ==)")

print("")
print("Checking any float safely:")
for value in [1.0, float("inf"), float("nan")]:
    print(f"   {str(value):<6} finite={math.isfinite(value):<6} "
          f"inf={math.isinf(value):<6} nan={math.isnan(value)}")

## Rounding, and banker's rounding

Python's `round()` uses **banker's rounding** — halves go to the nearest *even*
number. This surprises people, but it is deliberate: it avoids the upward bias
that always-round-up introduces over many values.

In [ ]:
print("Python rounds halves to the nearest EVEN number:")
print("")
for value in [0.5, 1.5, 2.5, 3.5, 4.5]:
    print(f"   round({value}) = {round(value)}")

print("")
print("Not a bug. Always rounding up would bias sums upward.")
print("This is the IEEE 754 default, and standard in finance.")

# The float representation adds a second layer of surprise.
print("")
print("And representation error can affect the result:")
print("   round(2.675, 2) =", round(2.675, 2), "<- expected 2.68")
print("   2.675 is really:", format(2.675, ".20f"))
print("   so it is just BELOW 2.675 and rounds down")

# Alternatives when you need different behaviour.
import math
from decimal import Decimal, ROUND_HALF_UP

print("")
print("Alternatives:")
print("   math.floor(2.7) ->", math.floor(2.7), " always down")
print("   math.ceil(2.1)  ->", math.ceil(2.1), " always up")
print("   math.trunc(-2.7)->", math.trunc(-2.7), "towards zero")
print("   int(-2.7)       ->", int(-2.7), "same as trunc")

# Decimal gives you explicit control.
exact = Decimal("2.675").quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
print("")
print("   Decimal with ROUND_HALF_UP ->", exact, "<- what you probably wanted")
print("")
print("Decimal is covered in 05.4.")

## When floats are the wrong tool

Floats are excellent for measurement and science. They are the **wrong choice**
for money.

In [ ]:
from decimal import Decimal

# A realistic money bug.
price = 0.1
quantity = 3

float_total = price * quantity
decimal_total = Decimal("0.1") * 3

print("Charging for 3 items at 0.10 each:")
print("   with float:  ", float_total)
print("   with Decimal:", decimal_total)
print("   float == 0.30?", float_total == 0.30)

print("")
print("One transaction is off by 0.00000000000000004. Across a million")
print("transactions that becomes a reconciliation problem.")

# The two common fixes.
print("")
print("FIX 1 - use Decimal for money (05.4):")
print("   Decimal('19.99') * 3 =", Decimal("19.99") * 3)

print("")
print("FIX 2 - store money as integer minor units (paise, cents):")
price_in_paise = 10
total_paise = price_in_paise * 3
print(f"   {price_in_paise} paise x 3 = {total_paise} paise = "
      f"{total_paise / 100:.2f} rupees")
print("   All arithmetic stays in exact integers.")

print("")
print("Use floats for: measurements, science, graphics, statistics.")
print("Never for: money, or anything requiring exact decimal results.")

## Takeaways

1. Floats are **binary** fractions following IEEE 754 — 1 sign bit, 11 exponent
   bits, 52 mantissa bits, giving about 15–17 significant digits.
2. `0.1` cannot be represented exactly in binary, just as `1/3` cannot in decimal.
   `0.1 + 0.2 == 0.30000000000000004` is correct behaviour.
3. Only fractions with **power-of-two denominators** are exact: `0.5`, `0.25`,
   `0.125`.
4. **Never compare floats with `==`.** Use `math.isclose()`, and pass `abs_tol`
   when either value may be zero.
5. Errors **accumulate**; `math.fsum()` avoids this for summation.
6. Subtracting nearly-equal numbers causes **catastrophic cancellation**.
7. `inf` and `nan` are real float values. `nan != nan`, so test with
   `math.isnan()`.
8. `round()` uses **banker's rounding** — halves go to even.
9. Never use floats for **money**. Use `Decimal` or integer minor units.

## Try it yourself

1. Print `format(0.1, '.20f')` and `(0.1).as_integer_ratio()`. What is stored?
2. Find three decimals that ARE exact in binary. What do their denominators
   share?
3. Add `0.1` a thousand times. How large is the error?
4. Compare `sum([0.1] * 10)` with `math.fsum([0.1] * 10)`.
5. Try `round(0.5)`, `round(1.5)`, `round(2.5)`. Explain the pattern.
6. Write a function comparing two floats safely, and test it against zero.